# 10 — Four edge-case neighborhoods

Expanding Clearwell with four more neighborhoods, each stress-testing one specific routing scenario: a construction closure, real dead ends, stairs with no explicit tag, and a genuinely unreachable island. Full automated tests live in `tests/test_edge_cases.py` — this notebook is for seeing each scenario run and visualizing the result.

### Load the expanded city

Clearwell now has 8 neighborhoods instead of 4 — same `load_clearwell()` call, no other code changes needed anywhere else in the project.

In [1]:
import sys
sys.path.append("..")

import networkx as nx
from src.data.synthetic_city import load_clearwell
from src.routing.pathfinding import find_accessible_route

G = load_clearwell()
print(G.number_of_nodes(), "nodes")
print(G.number_of_edges(), "edges")

126 nodes
360 edges


### Construction Zone: a direct route blocked by one closed segment

The `construction="yes"` tag on one segment should force a detour through the bypass row, even though the direct row is otherwise shorter.

In [2]:
plain = nx.shortest_path(G, "cz_0_0", "cz_0_4", weight="length")
accessible, cost = find_accessible_route(G, "cz_0_0", "cz_0_4")
print("plain shortest (walks through construction):", plain)
print("accessible route (detours around it):", accessible)

plain shortest (walks through construction): ['cz_0_0', 'cz_0_1', 'cz_0_2', 'cz_0_3', 'cz_0_4']
accessible route (detours around it): ['cz_0_0', 'cz_1_0', 'cz_1_1', 'cz_1_2', 'cz_1_3', 'cz_1_4', 'cz_0_4']


### Dead-End District: a through street with real cul-de-sacs

Routing end-to-end along the spine should never wander into a branch. Routing *into* a dead end should still work — reaching a branch's own dead-end node is a valid destination, even though nothing continues past it.

In [3]:
through_route, _ = find_accessible_route(G, "de_spine_0", "de_spine_4")
into_dead_end, _ = find_accessible_route(G, "de_spine_0", "de_branch_0_1")
print("through route (ignores branches):", through_route)
print("route into a dead end:", into_dead_end)

through route (ignores branches): ['de_spine_0', 'de_spine_1', 'de_spine_2', 'de_spine_3', 'de_spine_4']
route into a dead end: ['de_spine_0', 'de_spine_1', 'de_branch_0_0', 'de_branch_0_1']


### Stairs-Only Zone: inaccessible even with no tag at all

The stairs edge has `highway="steps"` and **no `wheelchair` tag whatsoever** — this is the real-world case where nobody bothered to tag it, but it's still obviously not wheelchair-accessible. The weighting function's `highway=="steps"` rule (added specifically for this) should route around it via the tagged ramp.

In [4]:
plain = nx.shortest_path(G, "so_bottom", "so_top", weight="length")
accessible, cost = find_accessible_route(G, "so_bottom", "so_top")
print("plain shortest (the untagged stairs):", plain)
print("accessible route (the tagged ramp):", accessible)

plain shortest (the untagged stairs): ['so_bottom', 'so_top']
accessible route (the tagged ramp): ['so_bottom', 'so_ramp_mid', 'so_top']


### Disconnected Island: a real "no path exists" case

Four nodes with zero edges to the rest of the city. `find_accessible_route` doesn't hide this with a fake fallback — it lets `networkx`'s own `NetworkXNoPath` propagate, so the caller knows for certain that no route exists rather than silently getting something wrong.

In [5]:
try:
    find_accessible_route(G, "di_0", "ot_0_0")
except nx.NetworkXNoPath as e:
    print("correctly raised NetworkXNoPath:", e)

# but the island's own nodes can still reach each other
path, cost = find_accessible_route(G, "di_0", "di_3")
print("within the island:", path)

correctly raised NetworkXNoPath: No path between di_0 and ot_0_0.
within the island: ['di_0', 'di_1', 'di_2', 'di_3']
